## Import libs

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.python.data import Dataset
from tensorflow.keras.optimizers import Adam
import seaborn as sns


from falsb4mpa.modeling.zhang.learning.multi_adv import train_loop as zhang_train
from falsb4mpa.dataset.load_data import load_data
from falsb4mpa.evaluation.evaluation import compute_predictive_metrics, compute_fair_metrics, compute_adv_metrics, compute_tradeoff, fair_evaluation, compute_intersectional_fair_metrics
from falsb4mpa.modeling.zhang.models.multi_adv import ZhangMultAdv

## Preliminaries

In [2]:
batch_size = 64
epochs = 100
learning_rate = 0.001

In [ ]:
cv_seeds = [13]
# cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'adult-mpa-bin-wout-agg'

In [5]:
x, y, a1, a2 = load_data(data_name)
raw_data = (x, y, a1, a2)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
a1dim = a1.shape[1]
a2dim = a2.shape[1]
zdim = 8
print(xdim, ydim, a1dim, a2dim, zdim)

97 1 1 1 8


## Result file

In [ ]:
header = [
    "model_name", "cv_seed", 
    "clas_acc", "f1-micro", "f1-macro",
    "a1_dp", "a1_deqodds", "a1_deqopp", 
    "a1_TN_g0", "a1_FP_g0", "a1_FN_g0", "a1_TP_g0", "a1_TN_g1", "a1_FP_g1", "a1_FN_g1", "a1_TP_g1", 
    "a2_dp", "a2_deqodds", "a2_deqopp", 
    "a2_TN_g0", "a2_FP_g0", "a2_FN_g0", "a2_TP_g0", "a2_TN_g1", "a2_FP_g1", "a2_FN_g1", "a2_TP_g1",
    "wc_spd", "wc_aod", "wc_eod",
    "last_cosine_similarity"
]

results = []

: 

## Testing

### DemPar

In [ ]:
fairdef = "DemPar"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4DP', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-13 18:18:46.187323: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.38/0.72 | Adv1 loss/acc 0.73/0.67 | Adv2 loss/acc 0.52/0.86 | Cos Sim -0.04


2026-05-13 18:19:01.426857: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.35/0.83 | Adv1 loss/acc 0.81/0.67 | Adv2 loss/acc 0.54/0.86 | Cos Sim 0.03
> Epoch: 3 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.89/0.67 | Adv2 loss/acc 0.57/0.86 | Cos Sim 0.08


2026-05-13 18:19:32.008073: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 64 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.09/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.19
> Epoch: 65 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.09/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.19
> Epoch: 66 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.10/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.19
> Epoch: 67 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.10/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.19
> Epoch: 68 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.10/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 69 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.10/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 70 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.11/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 71 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.11/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 72 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.11/0.67 | Adv2 loss/acc 1.16/0.86 | Cos Sim 0.19
> Epoch: 73 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.11/0.67 |

2026-05-13 18:51:04.054584: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.66/0.84 | Adv1 loss/acc 1.52/0.67 | Adv2 loss/acc 0.44/0.86 | Cos Sim 0.20
> Epoch: 28 | Clf loss/acc 0.67/0.84 | Adv1 loss/acc 1.53/0.67 | Adv2 loss/acc 0.44/0.86 | Cos Sim 0.20
> Epoch: 29 | Clf loss/acc 0.68/0.84 | Adv1 loss/acc 1.53/0.67 | Adv2 loss/acc 0.44/0.86 | Cos Sim 0.20
> Epoch: 30 | Clf loss/acc 0.69/0.84 | Adv1 loss/acc 1.54/0.67 | Adv2 loss/acc 0.45/0.86 | Cos Sim 0.20
> Epoch: 31 | Clf loss/acc 0.69/0.84 | Adv1 loss/acc 1.55/0.67 | Adv2 loss/acc 0.45/0.86 | Cos Sim 0.20
> Epoch: 32 | Clf loss/acc 0.70/0.84 | Adv1 loss/acc 1.55/0.67 | Adv2 loss/acc 0.45/0.86 | Cos Sim 0.20
> Epoch: 33 | Clf loss/acc 0.71/0.84 | Adv1 loss/acc 1.56/0.67 | Adv2 loss/acc 0.45/0.86 | Cos Sim 0.20
> Epoch: 34 | Clf loss/acc 0.71/0.84 | Adv1 loss/acc 1.56/0.67 | Adv2 loss/acc 0.45/0.86 | Cos Sim 0.20
> Epoch: 35 | Clf loss/acc 0.72/0.84 | Adv1 loss/acc 1.57/0.67 | Adv2 loss/acc 0.45/0.86 | Cos Sim 0.20
> Epoch: 36 | Clf loss/acc 0.72/0.84 | Adv1 loss/acc 1.57/0.67 |

2026-05-13 19:28:31.471708: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 55 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 2.19/0.68 | Adv2 loss/acc 0.59/0.86 | Cos Sim 0.28
> Epoch: 56 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 2.20/0.68 | Adv2 loss/acc 0.59/0.86 | Cos Sim 0.28
> Epoch: 57 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 2.20/0.68 | Adv2 loss/acc 0.59/0.86 | Cos Sim 0.28
> Epoch: 58 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 2.21/0.68 | Adv2 loss/acc 0.59/0.86 | Cos Sim 0.28
> Epoch: 59 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 2.21/0.68 | Adv2 loss/acc 0.60/0.86 | Cos Sim 0.28
> Epoch: 60 | Clf loss/acc 0.52/0.83 | Adv1 loss/acc 2.22/0.68 | Adv2 loss/acc 0.60/0.86 | Cos Sim 0.28
> Epoch: 61 | Clf loss/acc 0.53/0.83 | Adv1 loss/acc 2.22/0.68 | Adv2 loss/acc 0.60/0.86 | Cos Sim 0.28
> Epoch: 62 | Clf loss/acc 0.53/0.83 | Adv1 loss/acc 2.23/0.68 | Adv2 loss/acc 0.60/0.86 | Cos Sim 0.28
> Epoch: 63 | Clf loss/acc 0.53/0.83 | Adv1 loss/acc 2.23/0.68 | Adv2 loss/acc 0.60/0.86 | Cos Sim 0.28
> Epoch: 64 | Clf loss/acc 0.53/0.83 | Adv1 loss/acc 2.24/0.68 |

### EqOdds

In [ ]:
fairdef = "EqOdds"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOdds', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-13 11:52:21.044242: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.38/0.72 | Adv1 loss/acc 0.72/0.67 | Adv2 loss/acc 0.53/0.86 | Cos Sim -0.03


2026-05-13 11:52:36.767360: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.35/0.83 | Adv1 loss/acc 0.80/0.67 | Adv2 loss/acc 0.55/0.86 | Cos Sim 0.03
> Epoch: 3 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.87/0.67 | Adv2 loss/acc 0.57/0.86 | Cos Sim 0.07


2026-05-13 11:53:08.274766: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 64 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.06/0.67 | Adv2 loss/acc 1.13/0.86 | Cos Sim 0.18
> Epoch: 65 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.06/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.18
> Epoch: 66 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.07/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.18
> Epoch: 67 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.07/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.18
> Epoch: 68 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.07/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim 0.18
> Epoch: 69 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.08/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.18
> Epoch: 70 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.08/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 71 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.08/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 72 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.08/0.67 | Adv2 loss/acc 1.15/0.86 | Cos Sim 0.19
> Epoch: 73 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.09/0.67 |

2026-05-13 14:15:00.701855: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 128 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.20/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 129 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.20/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 130 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.20/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 131 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.20/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 132 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.20/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 133 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.20/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 134 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.21/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 135 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.21/0.67 | Adv2 loss/acc 1.24/0.86 | Cos Sim 0.22
> Epoch: 136 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.21/0.67 | Adv2 loss/acc 1.25/0.86 | Cos Sim 0.22
> Epoch: 137 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2

2026-05-13 14:54:02.744022: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 257 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 258 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 259 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 260 | Clf loss/acc 0.41/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 261 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 262 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 263 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 264 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 265 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.28/0.86 | Cos Sim 0.23
> Epoch: 266 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2

### EqOpp

In [ ]:
fairdef = "EqOpp"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1 = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOpp', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] + a1_metrics_g0 + a1_metrics_g1 
    result += [a2_dp, a2_deqodds, a2_deqopp] + a2_metrics_g0 + a2_metrics_g1
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp, a2_metrics_g0, a2_metrics_g1)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

/Users/lffpl/Projects/falsb4mpa/.venv/lib/python3.11/site-packages/keras/src/optimizers/base_optimizer.py:731: UserWarning: Gradients do not exist for variables ['U:0', 'c:0'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(
2026-05-05 18:46:31.541957: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.75/0.25 | Adv1 loss/acc 0.12/0.67 | Adv2 loss/acc 0.12/0.86


2026-05-05 18:46:50.166888: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.58/0.62 | Adv1 loss/acc 0.12/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 3 | Clf loss/acc 0.50/0.75 | Adv1 loss/acc 0.12/0.67 | Adv2 loss/acc 0.12/0.86


2026-05-05 18:47:27.694017: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 16 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 17 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 18 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 19 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 20 | Clf loss/acc 0.34/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 21 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 22 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 23 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.86
> Epoch: 24 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.85
> Epoch: 25 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.83
> Epoch: 26 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.10/0.67 | Adv2 loss/acc 0.12/0.83
> Epoch: 27 | Clf los

2026-05-05 18:56:13.408644: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 32 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.65 | Adv2 loss/acc 0.12/0.72
> Epoch: 33 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.65 | Adv2 loss/acc 0.12/0.69
> Epoch: 34 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.64 | Adv2 loss/acc 0.12/0.67
> Epoch: 35 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.64 | Adv2 loss/acc 0.12/0.66
> Epoch: 36 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.63 | Adv2 loss/acc 0.12/0.64
> Epoch: 37 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.63 | Adv2 loss/acc 0.12/0.62
> Epoch: 38 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.62 | Adv2 loss/acc 0.12/0.61
> Epoch: 39 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.62 | Adv2 loss/acc 0.12/0.60
> Epoch: 40 | Clf loss/acc 0.33/0.83 | Adv1 loss/acc 0.11/0.62 | Adv2 loss/acc 0.12/0.58
> Epoch: 41 | Clf loss/acc 0.33/0.84 | Adv1 loss/acc 0.11/0.61 | Adv2 loss/acc 0.12/0.57
> Epoch: 42 | Clf loss/acc 0.33/0.84 | Adv1 loss/acc 0.11/0.61 | Adv2 loss/acc 0.12/0.55
> Epoch: 43 | Clf los

2026-05-05 19:06:15.184918: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 64 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.56 | Adv2 loss/acc 0.12/0.43
> Epoch: 65 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.43
> Epoch: 66 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 67 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 68 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 69 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 70 | Clf loss/acc 0.35/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 71 | Clf loss/acc 0.36/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 72 | Clf loss/acc 0.36/0.84 | Adv1 loss/acc 0.12/0.55 | Adv2 loss/acc 0.12/0.42
> Epoch: 73 | Clf loss/acc 0.36/0.84 | Adv1 loss/acc 0.12/0.54 | Adv2 loss/acc 0.12/0.42
> Epoch: 74 | Clf loss/acc 0.36/0.83 | Adv1 loss/acc 0.12/0.54 | Adv2 loss/acc 0.12/0.41
> Epoch: 75 | Clf los

## Saving into DF then CSV

In [ ]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,f1-micro,f1-macro,a1_dp,a1_deqodds,a1_deqopp,a1_TN_g0,a1_FP_g0,...,a2_FN_g0,a2_TP_g0,a2_TN_g1,a2_FP_g1,a2_FN_g1,a2_TP_g1,wc_spd,wc_aod,wc_eod,last_cosine_similarity
0,MultAdvBin4EqOdds,13,0.837604,0.837604,0.77471,0.796744,0.895753,0.896345,5459.0,830.0,...,1102.0,1942.0,1570.0,43.0,156.0,146.0,0.725926,0.775636,0.687961,0.229894


In [ ]:
result_df.to_csv(f'../../results/{data_name}-mult_adv-{epochs}.csv')